In [22]:
import pandas as pd

path = "/content/sample_data/Coachella-2015-2-DFE.csv"
df = pd.read_csv(path, encoding="latin1", low_memory=False)

df.head()

,coachella_sentiment,coachella_yn,name,retweet_count,text,tweet_coord,tweet_created,tweet_id,tweet_location,user_timezone
0,positive,yes,kokombil,0,#Coachella2015 tickets selling out in less tha...,"[0.0, 0.0]",1/7/15 15:02,5.529630e+17,NaN,Quito
1,positive,yes,MisssTaraaa10,2,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ÙÓ...,NaN,1/7/15 15:02,5.529630e+17,united states,NaN
2,positive,yes,NMcCracken805,0,#Coachella2015 #VIP passes secured! See you th...,NaN,1/7/15 15:01,5.529630e+17,"Costa Mesa, CA",NaN
3,positive,yes,wxpnfm,1,PhillyÛªs @warondrugsjams will play #Coachell...,NaN,1/7/15 15:01,5.529630e+17,"Philadelphia, PA and Worldwide",Quito
4,positive,yes,Caesears,0,If briana and her mom out to #Coachella2015 i...,NaN,1/7/15 15:00,5.529630e+17,NaN,NaN


In [23]:
import re

HASHTAG_RE = re.compile(r"(?i)(?<!\w)#\w+")  # hashtags like #coachella, #Coachella2015
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")

def extract_hashtags(text: str):
    if not isinstance(text, str):
        return []
    return HASHTAG_RE.findall(text)

def extract_emails(text: str):
    if not isinstance(text, str):
        return []
    return EMAIL_RE.findall(text)

df["hashtags"] = df["text"].apply(extract_hashtags)
df["emails"] = df["text"].apply(extract_emails)

df[["text", "hashtags", "emails"]].head(10)

,text,hashtags,emails
0,#Coachella2015 tickets selling out in less tha...,[#Coachella2015],[]
1,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ÙÓ...,[#Coachella2015],[]
2,#Coachella2015 #VIP passes secured! See you th...,"[#Coachella2015, #VIP]",[]
3,PhillyÛªs @warondrugsjams will play #Coachell...,"[#Coachella2015, #GovBall2015]",[]
4,If briana and her mom out to #Coachella2015 i...,[#Coachella2015],[]
5,West side is the best side!\n#west #coas #Coac...,"[#west, #coas, #Coachella2015]",[]
6,Coachella tickets are now sold out _Ù÷_ &amp; ...,"[#Coachella2015, #Coachella]",[]
7,#Coachella2015 I absolutely can NOT wait. This...,[#Coachella2015],[]
8,If someone got me to Coachella if be your frie...,"[#truth, #desprate, #Coachella2015, #Coachella]",[]
9,RT @brownjenjen:  Õ http://t.co/mxCREvIlGP 71...,[#Coachella2015],[]


In [25]:
import re

USERNAME_RE = re.compile(r"(?<!\w)@\w+")
LINK_RE = re.compile(r"https?://\S+|www\.\S+", re.I)
NON_ASCII_RE = re.compile(r"[^\x00-\x7F]+")
DIGITS_RE = re.compile(r"\d+")
MULTISPACE_RE = re.compile(r"\s+")

STOP_WORDS = {
    "the","a","an","and","or","is","are","was","were","to","of","in","on","at","for","with",
    "this","that","it","its","i","you","we","they","he","she","my","your","our","their",
    "be","been","being","as","by","from","but","not","so","if","then","than"
}

def remove_usernames(text: str) -> str:
    return USERNAME_RE.sub("", text)

def remove_links(text: str) -> str:
    return LINK_RE.sub("", text)

def remove_non_ascii_symbols(text: str) -> str:
    return NON_ASCII_RE.sub("", text)

def to_lower(text: str) -> str:
    return text.lower()

def remove_stop_words(text: str) -> str:
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return " ".join(tokens)

def remove_digits(text: str) -> str:
    return DIGITS_RE.sub("", text)

def remove_special_characters(text: str) -> str:
    # keep only letters + spaces
    return re.sub(r"[^A-Za-z\s]", " ", text)

def normalize_whitespace(text: str) -> str:
    return MULTISPACE_RE.sub(" ", text).strip()

def clean_tweet(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = remove_usernames(text)
    text = remove_links(text)
    text = remove_non_ascii_symbols(text)
    text = to_lower(text)
    text = remove_stop_words(text)
    text = remove_digits(text)
    text = remove_special_characters(text)
    text = normalize_whitespace(text)
    return text

df["clean_text"] = df["text"].apply(clean_tweet)

df[["text", "hashtags", "emails", "clean_text"]].head(10)


,text,hashtags,emails,clean_text
0,#Coachella2015 tickets selling out in less tha...,[#Coachella2015],[],coachella tickets selling out less minutes
1,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ÙÓ...,[#Coachella2015],[],rt wait absolute fire coachella
2,#Coachella2015 #VIP passes secured! See you th...,"[#Coachella2015, #VIP]",[],coachella vip passes secured see there bitches...
3,PhillyÛªs @warondrugsjams will play #Coachell...,"[#Coachella2015, #GovBall2015]",[],phillys will play coachella amp govball watch ...
4,If briana and her mom out to #Coachella2015 i...,[#Coachella2015],[],briana her mom out coachella im out them
5,West side is the best side!\n#west #coas #Coac...,"[#west, #coas, #Coachella2015]",[],west side best side west coas coachella
6,Coachella tickets are now sold out _Ù÷_ &amp; ...,"[#Coachella2015, #Coachella]",[],coachella tickets now sold out amp had opportu...
7,#Coachella2015 I absolutely can NOT wait. This...,[#Coachella2015],[],coachella absolutely can wait weekend about ex...
8,If someone got me to Coachella if be your frie...,"[#truth, #desprate, #Coachella2015, #Coachella]",[],someone got me coachella friend life truth des...
9,RT @brownjenjen:  Õ http://t.co/mxCREvIlGP 71...,[#Coachella2015],[],rt coachella coachella makes space rockers rav...


In [29]:
out_path = "/content/sample_data/cleaned.csv"
df.to_csv(out_path, index=False, encoding="utf-8")
out_path

'/content/sample_data/cleaned.csv'